In [29]:
tabla='cmace10'
dim='dim_subacti'

In [30]:
import oracledb
import pandas as pd
from sqlalchemy import create_engine
import sys
from sqlalchemy import create_engine
from sqlalchemy import text
import sys
from decouple import config

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="INDICADORES_ESSALUD"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena4  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine4 = create_engine(cadena4)
connection4 = engine4.connect()

In [31]:
oracledb.init_oracle_client()
oracledb.version = "8.3.0"
sys.modules["cx_Oracle"] = oracledb
un = config("USER4_BDI_POSTGRES")
pw = config("PASS4_BDI_POSTGRES")
hostname=config("HOST4_BDI_POSTGRES")
service_name="WNET"
port = 1527

engine2 = create_engine(f'oracle://{un}:{pw}@',connect_args={
        "host": hostname,
        "port": port,
        "service_name": service_name
    }
)

connection2 = engine2.connect()

base2 = pd.read_sql_query(f"SELECT * FROM {tabla}", con=connection2)

connection2.close()




In [32]:
base2

,actcod,actespcod,actespnom,tipoprogperscod,tipoparprocod,actespestregcod,actespmulcitflg,actespturno24h,actesptelecon
0,B1,659,OPERADOR DE PROCEDIMIENTOS,2,44,0,0,NaN,NaN
1,B2,772,ACTIVIDADES DE REHABILITACION,1,51,0,0,NaN,NaN
2,B2,773,REALIZACION DE RADIOGRAFIAS,2,None,0,0,NaN,NaN
3,B5,055,ATENCION DE TRABAJO SOCIAL,2,43,1,0,NaN,NaN
4,B5,088,VACACIONES,2,None,0,0,NaN,NaN
...,...,...,...,...,...,...,...,...,...
1679,96,039,DILATACION DE COARTACION DE AORTA CON BALON,1,11,1,0,NaN,NaN
1680,96,040,DILATACION CON PROTESIS DE ESTENOSIS DE RAMA P...,1,11,1,0,NaN,NaN
1681,96,041,VALVULOPLASTIA MITRAL CON BALON,1,11,1,0,NaN,NaN
1682,96,042,VALVULOPLASTIA MITRAL CON DOBLE BALON,1,11,1,0,NaN,NaN


In [33]:
base2.columns

Index(['actcod', 'actespcod', 'actespnom', 'tipoprogperscod', 'tipoparprocod',
       'actespestregcod', 'actespmulcitflg', 'actespturno24h',
       'actesptelecon'],
      dtype='object')

In [34]:

#################################################################################################################################################################################################################################################################################
#CREAMOS LA TABLA TEMPORAL Y LA SUBIMOS AL POSTGRES SQL

DB_USER=config("USER2_BDI_POSTGRES")
DB_PASSWORD=config("PASS2_BDI_POSTGRES")
DB_NAME="dl_essi"
DB_PORT="5432"
DB_HOST=config("HOST2_BDI_POSTGRES")
cadena3  = f"postgresql://{DB_USER}:{DB_PASSWORD}@{DB_HOST}:{DB_PORT}/{DB_NAME}"
engine3 = create_engine(cadena3)
connection3 = engine3.connect()


query_count_before = "SELECT COUNT(*) FROM cmace10"
cant_antes = connection3.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla cmace10 antes de la inserción: {cant_antes}")


#connection3.execute('CREATE TEMPORARY TABLE tmp_cmsho10 ()')
base2.to_sql(name='tmp_cmace10', con=engine3, if_exists='replace', index=False)

#################################################################################################################################################################################################################################################################################
#ACTUALIZAMOS EL cmsho10 FALSO CON LO OBTENIDO DEL ORACLE

query="""

ALTER TABLE tmp_cmace10
ALTER COLUMN actcod TYPE character(2),
ALTER COLUMN actespcod TYPE character(3),
ALTER COLUMN actespnom TYPE character(80),
ALTER COLUMN tipoprogperscod TYPE character(1),
ALTER COLUMN tipoparprocod TYPE character(2),
ALTER COLUMN actespestregcod TYPE character(1),
ALTER COLUMN actespmulcitflg TYPE character(1),
ALTER COLUMN actespturno24h TYPE numeric(1,0),
ALTER COLUMN actesptelecon TYPE numeric(1,0);



UPDATE cmace10 
SET 
actcod = t.actcod,
actespcod = t.actespcod,
actespnom = t.actespnom,
tipoprogperscod = t.tipoprogperscod,
tipoparprocod = t.tipoparprocod,
actespestregcod = t.actespestregcod,
actespmulcitflg = t.actespmulcitflg,
actespturno24h = t.actespturno24h,
actesptelecon = t.actesptelecon  

FROM tmp_cmace10 t 
WHERE cmace10.actcod = t.actcod
and cmace10.actespcod = t.actespcod
and cmace10.actcod IS NOT NULL;

INSERT INTO cmace10 
(actcod,actespcod,actespnom,tipoprogperscod,tipoparprocod,actespestregcod,actespmulcitflg,actespturno24h,actesptelecon) 
SELECT 
actcod,actespcod,actespnom,tipoprogperscod,tipoparprocod,actespestregcod,actespmulcitflg,actespturno24h,actesptelecon

FROM tmp_cmace10  t
WHERE NOT EXISTS (
    SELECT 1 
    FROM cmace10
    WHERE cmace10.actcod = t.actcod
    and cmace10.actespcod = t.actespcod
    and cmace10.actcod IS NOT NULL
) ;

"""

c1= text(query)
connection3.execute(c1)

#BORRAMOS LAS TABLAS
query2="""
DROP TABLE tmp_cmace10;
DELETE FROM cmace10 WHERE actespcod ='';
SELECT COUNT(*) FROM cmace10;
"""
c2= text(query2)
cursor=connection3.execute(c2)
cant_despues = cursor.fetchone()[0]
print(f"Cantidad de filas en la tabla cmace10 después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")
base2 = pd.read_sql_query(f"SELECT * FROM cmace10", con=connection3)

connection3.close()

Cantidad de filas en la tabla cmace10 antes de la inserción: 1684
Cantidad de filas en la tabla cmace10 después de la inserción: 1684
La cantidad de filas insertadas fue de: 0


In [35]:
#SUBIMOS ESA TABLA ACTUALIZADA COMO TEMPORAL A LA BASE DEL DIM ACTIVIT
base2.to_sql(name=f'tmp_{tabla}', con=engine4, if_exists='replace', index=False)

#ACTUALIZAMOS EL DIM ACTIVITI FALSO CON LA BASE DE DATOS QUE SUBIMOS
query_count_before = f"SELECT COUNT(*) FROM {dim}"
cant_antes = connection4.execute(query_count_before).fetchone()[0]
print(f"Cantidad de filas en la tabla {dim} antes de la inserción: {cant_antes}")

query=f"""

ALTER TABLE tmp_{tabla} 
ALTER COLUMN actcod TYPE character(2),
ALTER COLUMN actespcod TYPE character(3),
ALTER COLUMN actespnom TYPE character(80),
ALTER COLUMN tipoprogperscod TYPE character(1),
ALTER COLUMN tipoparprocod TYPE character(2),
ALTER COLUMN actespestregcod TYPE character(1),
ALTER COLUMN actespmulcitflg TYPE character(1),
ALTER COLUMN actespturno24h TYPE numeric(1,0),
ALTER COLUMN actesptelecon TYPE numeric(1,0);
INSERT INTO {dim} 

(cod_sub,des_sub,cod_act) 

SELECT 


actespcod,actespnom,actcod

FROM tmp_{tabla} t
WHERE NOT EXISTS (
    SELECT 1 
    FROM {dim} d 
    WHERE 
    
    d.cod_sub = t.actespcod
    AND d.cod_act = t.actcod
);
"""

c1= text(query)
cursor=connection4.execute(c1)

#BORRAMOS LAS TABLAS
query2=f"""
DROP TABLE tmp_{tabla};
SELECT COUNT(*) FROM {dim};
"""

c2= text(query2)
cursor=connection4.execute(c2)
cant_despues = cursor.fetchone()[0]
print(f"Cantidad de filas en la tabla {dim} después de la inserción: {cant_despues}")
print(f"La cantidad de filas insertadas fue de: {cant_despues-cant_antes}")

Cantidad de filas en la tabla dim_subacti antes de la inserción: 1684
Cantidad de filas en la tabla dim_subacti después de la inserción: 1684
La cantidad de filas insertadas fue de: 0
